In [1]:
import sys
import os
from pathlib import Path
from typing import Optional
# sys.path.insert(0, str(Path(__file__).parent.parent))
sys.path.append(os.path.abspath("../"))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

pd.set_option('display.max_columns', None)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("pipeline")

from sqlalchemy import (
    create_engine, text
)
from sqlalchemy.pool import StaticPool

from pipeline import load_features_for_ml, get_feature_columns
from models.nfl_forecaster import NFLPerformanceForecaster
from models.injury_risk import InjuryRiskModel
from models.college_translator import CollegeToNFLTranslator
from registry.model_store import ModelStore


In [2]:
current_path = os.getcwd()
DB_PATH = str(Path(current_path).parent / "dynasty_scout.db")
DB_PATH

'/Users/sandeeptiwari/Desktop/dynasty-ai-engine/dynasty_scout.db'

## Loading Data

In [3]:
df_all = load_features_for_ml(
    min_season=2015,          # 2016+ for NGS features
    max_season=2024,          # hold out 2024 for evaluation
    require_target=True,
    player_type="nfl",
)
df_all

00:32:45 [INFO] pipeline: Loaded 2862 rows for ML training (position=all, seasons=2015–2024)


,id,player_id,season,position,player_type,fantasy_ppg_ppr,fantasy_ppg_last_season,fantasy_ppg_2yr_avg,fantasy_ppg_3yr_avg,fantasy_ppg_trend,games_played,career_games,target_share,air_yards_share,wopr,snap_pct,snap_pct_trend,carries_per_game,targets_per_game,yards_per_target,yards_per_carry,yards_after_catch_per_rec,racr,epa_per_play,cpoe,ryoe_per_att,separation_avg,avg_yac_above_expectation,games_missed_last_season,games_missed_2yr_total,injury_risk_score,soft_tissue_injury_flag,acl_history_flag,concussion_history_count,injury_designation_count,age,age_at_nfl_entry,years_experience,age_vs_position_peak,team_pass_rate,team_pass_rate_neutral,team_plays_per_game,team_pass_attempts,team_points_per_game,offensive_line_rank,new_team_flag,new_oc_flag,scheme_fit_score,dominator_rating,breakout_age,college_yards_per_game,college_tds_per_game,college_conference_tier,draft_round,draft_pick_normalized,sparq_score,relative_athletic_score,speed_score,height_weight_bmi,forty_yard,vertical_jump,fantasy_ppg_next_season,player_name,nfl_team,sleeper_id
2,3,00-0019596,2015,QB,nfl,21.543750,NaN,NaN,NaN,NaN,16.0,NaN,0.017857,0.012259,0.035367,0.989444,NaN,2.125000,0.062500,36.00000,1.558824,29.000000,5.142857,3.129031,NaN,NaN,NaN,NaN,0,0,0.000000,0,0,0,1,38.078029,23.080082,23,4.078029,0.6503,0.6693,65.78,770.0,29.06,11.0,0,0,0.50000,NaN,NaN,NaN,NaN,NaN,6.0,0.071966,NaN,NaN,NaN,NaN,NaN,NaN,21.546667,Tom Brady,None,None
3,4,00-0019596,2016,QB,nfl,21.546667,21.543750,21.543750,21.543750,NaN,12.0,16.0,0.000000,0.000000,0.000000,0.972667,-0.016778,2.333333,0.000000,NaN,2.285714,NaN,NaN,NaN,3.269891,NaN,NaN,NaN,0,0,0.000000,0,0,0,4,39.080082,23.080082,23,5.080082,0.5640,0.5969,67.84,727.0,27.56,6.0,0,0,0.50000,NaN,NaN,NaN,NaN,NaN,6.0,0.071966,NaN,NaN,NaN,NaN,NaN,NaN,18.492500,Tom Brady,None,None
4,5,00-0019596,2017,QB,nfl,18.492500,21.546667,21.545208,21.545208,0.002917,16.0,28.0,0.000000,0.000000,0.000000,0.980526,0.007860,1.562500,0.000000,NaN,1.120000,NaN,NaN,NaN,0.849809,NaN,NaN,NaN,0,0,0.000000,0,0,0,1,40.079398,23.080082,23,6.079398,0.5984,0.6057,67.63,769.0,28.62,9.0,0,0,0.50000,NaN,NaN,NaN,NaN,NaN,6.0,0.071966,NaN,NaN,NaN,NaN,NaN,NaN,17.581250,Tom Brady,None,None
5,6,00-0019596,2018,QB,nfl,17.581250,18.492500,20.019583,20.527639,-1.525625,16.0,44.0,0.025000,0.000000,0.037500,1.000000,0.019474,1.437500,0.062500,6.00000,1.521739,6.000000,NaN,-0.220586,-1.562972,NaN,NaN,NaN,0,0,0.000000,0,0,0,1,41.078713,23.080082,23,7.078713,0.5633,0.5658,66.89,716.0,32.94,6.0,1,1,0.50000,NaN,NaN,NaN,NaN,NaN,6.0,0.071966,NaN,NaN,NaN,NaN,NaN,NaN,16.480000,Tom Brady,None,None
6,7,00-0019596,2019,QB,nfl,16.480000,17.581250,18.036875,19.206806,-1.982708,16.0,60.0,0.000000,0.000000,0.000000,0.988235,-0.011765,1.625000,0.000000,NaN,1.307692,NaN,NaN,NaN,-2.997041,NaN,NaN,NaN,0,0,0.000000,0,0,0,1,42.078029,23.080082,23,8.078029,0.5933,0.6040,68.12,687.0,26.25,3.0,1,1,0.50000,NaN,NaN,NaN,NaN,NaN,6.0,0.071966,NaN,NaN,NaN,NaN,NaN,NaN,21.120000,Tom Brady,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5487,5488,00-0039146,2023,WR,nfl,13.575000,NaN,6.836364,6.836364,NaN,16.0,NaN,2.827228,3.255236,6.519507,0.558889,NaN,0.687500,5.875000,8.43617,10.818182,5.234375,0.817526,0.354247,NaN,NaN,3.444668,0.284881,0,0,0.000000,0,0,0,3,23.342916,23.342916,4,-2.657084,0.5707,0.5354,61.79,670.0,22.53,3.0,0,0,1.00000,0.340734,20.344969,78.923077,0.769231,1.0,2.0,0.314135,70.645825,9.468991,95.374343,26.078357,4.45,33.5,11.588235,Jayden Reed,None,None
5489,5490,00-0039150,2023,QB,nfl,9.773750,NaN,13.575000,13.575000,NaN,16.0,NaN,0.000000,0.000000,0.000000,0.998125,NaN,2.437500,0.000000,NaN,6.487179,NaN,NaN,NaN,-1.574166,NaN,NaN,NaN,1,0,0.020588,0,0,0,1,22.102669,22.102669,4,-11.897331,0.5933,0.5399,64.65,652.0,13.88,30.0,0,0,0.50000,0.000000,NaN,22.285714,0.142857,1.0,

In [46]:
print(f"  {len(df_all)} player-seasons loaded")
print(f"  Positions: {df_all['position'].value_counts().to_dict()}")
print(f"  Seasons: {sorted(df_all['season'].unique())}")
print(f"\nTarget variable distribution:")
print(df_all["fantasy_ppg_next_season"].describe().round(2))

  2862 player-seasons loaded
  Positions: {'WR': 1158, 'RB': 757, 'TE': 610, 'QB': 337}
  Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

Target variable distribution:
count    2862.00
mean        8.65
std         5.59
min        -1.40
25%         4.15
50%         7.67
75%        12.39
max        30.13
Name: fantasy_ppg_next_season, dtype: float64


## Data Quality
___
### NFL features

In [47]:
feat_config = get_feature_columns()
features_to_check = feat_config["nfl_features"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\nNFL Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())


NFL Feature missingness:
                      feature  missing_pct
0    catch_pct_above_expected     1.000000
1                        cpoe     0.904263
2                ryoe_per_att     0.899022
3              separation_avg     0.677498
4           fantasy_ppg_trend     0.489867
5             yards_per_carry     0.363033
6              snap_pct_trend     0.278826
7     fantasy_ppg_last_season     0.269043
8                career_games     0.269043
9                        racr     0.101328
10  yards_after_catch_per_rec     0.099581
11           yards_per_target     0.095038
12               epa_per_play     0.095038
13        fantasy_ppg_2yr_avg     0.052061
14     team_pass_rate_neutral     0.035290
15             team_pass_rate     0.035290
16        offensive_line_rank     0.035290
17       team_points_per_game     0.035290
18         team_pass_attempts     0.035290
19        team_plays_per_game     0.035290
20        fantasy_ppg_3yr_avg     0.014675
21                   snap_pc

### College Features

In [19]:
feat_config = get_feature_columns()
features_to_check = feat_config["college_features"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\nCollege Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())


College Feature missingness:
                    feature  missing_pct
0              breakout_age     0.714535
1               sparq_score     0.705101
2             vertical_jump     0.577568
3               speed_score     0.551712
4                forty_yard     0.549965
5    college_yards_per_game     0.547170
6      college_tds_per_game     0.547170
7   relative_athletic_score     0.500000
8          dominator_rating     0.387142
9   college_conference_tier     0.357442
10    draft_pick_normalized     0.215584
11              draft_round     0.211041
12                      age     0.000000
13         age_at_nfl_entry     0.000000
14         years_experience     0.000000
15     age_vs_position_peak     0.000000


### Performance Features

In [20]:
feat_config = get_feature_columns()
features_to_check = feat_config["performance"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\Performance Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

\Performance Feature missingness:
                   feature  missing_pct
0        fantasy_ppg_trend     0.489867
1  fantasy_ppg_last_season     0.269043
2             career_games     0.269043
3      fantasy_ppg_2yr_avg     0.052061
4      fantasy_ppg_3yr_avg     0.014675
5             games_played     0.000000


### Role Features

In [21]:
feat_config = get_feature_columns()
features_to_check = feat_config["role"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\Role Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

\Role Feature missingness:
            feature  missing_pct
0    snap_pct_trend     0.278826
1          snap_pct     0.009434
2      target_share     0.000000
3   air_yards_share     0.000000
4              wopr     0.000000
5  carries_per_game     0.000000
6  targets_per_game     0.000000


### Efficiency

In [22]:
feat_config = get_feature_columns()
features_to_check = feat_config["efficiency"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\Efficiency Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

\Efficiency Feature missingness:
                     feature  missing_pct
0   catch_pct_above_expected     1.000000
1                       cpoe     0.904263
2               ryoe_per_att     0.899022
3             separation_avg     0.677498
4            yards_per_carry     0.363033
5                       racr     0.101328
6  yards_after_catch_per_rec     0.099581
7           yards_per_target     0.095038
8               epa_per_play     0.095038


### Injury

In [23]:
feat_config = get_feature_columns()
features_to_check = feat_config["injury"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\Injury Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

\Injury Feature missingness:
                    feature  missing_pct
0  games_missed_last_season          0.0
1    games_missed_2yr_total          0.0
2         injury_risk_score          0.0
3   soft_tissue_injury_flag          0.0
4          acl_history_flag          0.0
5  concussion_history_count          0.0


### Age Curve

In [24]:
feat_config = get_feature_columns()
features_to_check = feat_config["age_curve"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\Age Curve Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

\Age Curve Feature missingness:
                feature  missing_pct
0                   age          0.0
1      age_at_nfl_entry          0.0
2      years_experience          0.0
3  age_vs_position_peak          0.0


### Team Context

In [25]:
feat_config = get_feature_columns()
features_to_check = feat_config["team_context"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\Team Context Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

\Team Context Feature missingness:
                  feature  missing_pct
0          team_pass_rate      0.03529
1  team_pass_rate_neutral      0.03529
2     team_plays_per_game      0.03529
3      team_pass_attempts      0.03529
4    team_points_per_game      0.03529
5     offensive_line_rank      0.03529
6           new_team_flag      0.00000
7             new_oc_flag      0.00000
8        scheme_fit_score      0.00000


### Prospect

In [26]:
feat_config = get_feature_columns()
features_to_check = feat_config["prospect"]
missing_pct = (
    df_all[features_to_check]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\Prospect Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

\Prospect Feature missingness:
                    feature  missing_pct
0              breakout_age     0.714535
1               sparq_score     0.705101
2             vertical_jump     0.577568
3               speed_score     0.551712
4                forty_yard     0.549965
5    college_yards_per_game     0.547170
6      college_tds_per_game     0.547170
7   relative_athletic_score     0.500000
8          dominator_rating     0.387142
9   college_conference_tier     0.357442
10    draft_pick_normalized     0.215584
11              draft_round     0.211041


### Target

In [27]:
feat_config = get_feature_columns()
features_to_check = feat_config["target"]
missing_pct = (
    df_all[[features_to_check]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
print("\Target Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

\Target Feature missingness:
                   feature  missing_pct
0  fantasy_ppg_next_season          0.0


### Train Forecaster

In [48]:
wr_model = NFLPerformanceForecaster("WR")
wr_metrics = wr_model.train_with_tracking(
    df_all,
    params={"position": "WR", "model": "LightGBM"}
)
print(f"\nMetrics:")
for k, v in wr_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: catch_pct_above_expected: object

In [49]:
df_all.isna().mean().sort_values(ascending=False)[:10]

sleeper_id                  1.000000
catch_pct_above_expected    1.000000
height_weight_bmi           1.000000
nfl_team                    1.000000
cpoe                        0.904263
ryoe_per_att                0.899022
breakout_age                0.714535
sparq_score                 0.705101
separation_avg              0.677498
vertical_jump               0.577568
dtype: float64